In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# COCO keypoint format
COCO_KEYPOINTS = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

class PoseNet(nn.Module):
    """
    Simple pose estimation network that outputs heatmaps for each keypoint
    """
    def __init__(self, num_keypoints=17):
        super(PoseNet, self).__init__()
        
        # Backbone
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        
        # Heatmap prediction
        self.heatmap_conv = nn.Conv2d(256, num_keypoints, kernel_size=1)
        
        self.num_keypoints = num_keypoints
    
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        heatmaps = self.heatmap_conv(x)
        return heatmaps

def extract_keypoints(heatmaps):
    """
    Extract keypoint coordinates from heatmaps
    """
    batch_size, num_keypoints, height, width = heatmaps.shape
    keypoints = []
    
    for b in range(batch_size):
        person_keypoints = []
        for k in range(num_keypoints):
            heatmap = heatmaps[b, k].detach().cpu().numpy()
            y, x = np.unravel_index(heatmap.argmax(), heatmap.shape)
            confidence = heatmap[y, x]
            person_keypoints.append([x, y, confidence])
        keypoints.append(person_keypoints)
    
    return keypoints

pose_net = PoseNet(num_keypoints=17).to(device)
print("PoseNet created successfully")

## 1. Heatmap-Based Pose Estimation

# Pose Estimation
## Detecting Human Body Keypoints and Poses

Topics covered:
- Single-person pose estimation (SPPEs)
- Multi-person pose estimation (MPPEs)
- Heatmap regression for keypoint detection
- Part Affinity Fields (PAF)
- OpenPose architecture
- HRNet (High-Resolution Network)
- Video pose tracking and smoothing
- Action recognition from poses